# Catalog Score — Esplorazione

**Obiettivo**: valutare se un punteggio numerico composito a livello di fonte/catalogo ha senso, 
e se discrimina bene tra fonti "ricche" e "deboli" nel Source Observatory.

**Domande guida**:
1. Fonti diverse hanno profili radicalmente diversi o sono comparabili?
2. Ci sono metriche dell'inventario che correlano con la qualità percepita?
3. Un catalog score composito sarebbe stabile o ballerino?
4. E se usiamo i dati già disponibili senza `source_check_results` (che è quasi vuoto)?

**Data**: 2026-06-07

In [ ]:
import json

import numpy as np
import pandas as pd

INVENTORY_PATH = "../data/catalog_inventory/generated/catalog_inventory_latest.parquet"
RADAR_PATH = "../data/radar/radar_summary.json"
SIGNALS_PATH = "../data/catalog/catalog_signals.json"
REPORT_PATH = "../data/catalog_inventory/generated/catalog_inventory_report.json"

df = pd.read_parquet(INVENTORY_PATH)
with open(RADAR_PATH) as f:
    radar = json.load(f)
with open(SIGNALS_PATH) as f:
    signals = json.load(f)
with open(REPORT_PATH) as f:
    report = json.load(f)

print(f"Inventory: {len(df)} rows, {df['source_id'].nunique()} sources")
print(f"Radar: {radar['sources_total']} sources")
print(f"Signals: {signals['sources_checked']} sources")

---
## 1. Struttura inventario: righe per fonte e protocollo

In [ ]:
# Righe per fonte
src_counts = df.groupby(["source_id", "protocol"]).size().reset_index(name="items_raw")
src_counts = src_counts.sort_values("items_raw", ascending=False)
print("=== Item (righe) per fonte ===")
display(src_counts)

# Totale per protocollo
print("\n=== Per protocollo ===")
display(df["protocol"].value_counts().to_frame("count"))

---
## 2. Il problema HTML: righe vs dataset reali

Nell'inventario HTML ogni **file** è una riga, ma più file appartengono allo
stesso dataset (prefisso comune). Un prefisso `INFANZIA` con 64 file = 1 dataset
con 64 variazioni (anni, regioni).

Se usiamo righe grezze, le fonti HTML vengono sovrarappresentate.
Dobbiamo aggregare per `prefix`.

In [ ]:
html = df[df["protocol"] == "html"].copy()

# Per HTML: dataset_reali = prefix unici
html_datasets = html.groupby("source_id")["prefix"].nunique().reset_index(name="datasets_real")
html_rows = html.groupby("source_id").size().reset_index(name="items_raw")
html_stats = html_rows.merge(html_datasets, on="source_id")
html_stats["inflation_ratio"] = (html_stats["items_raw"] / html_stats["datasets_real"]).round(1)
print("=== Fonti HTML: righe vs dataset reali ===")
display(html_stats.sort_values("items_raw", ascending=False))

In [ ]:
# Per CKAN/SDMX/SPARQL: ogni item è un dataset reale
non_html = df[df["protocol"] != "html"].copy()
non_html_datasets = non_html.groupby("source_id").size().reset_index(name="datasets_real")
non_html_datasets["items_raw"] = non_html_datasets["datasets_real"]
non_html_datasets["inflation_ratio"] = 1.0

# Unisco tutto
all_datasets = pd.concat(
    [
        html_stats[["source_id", "items_raw", "datasets_real", "inflation_ratio"]],
        non_html_datasets[["source_id", "items_raw", "datasets_real", "inflation_ratio"]],
    ]
)
all_datasets = all_datasets.sort_values("datasets_real", ascending=False)

print("=== Dataset REALI per fonte (dopo aggregazione HTML) ===")
display(all_datasets)
print(f"\nTotale righe inventory: {df.shape[0]}")
print(f"Totale dataset reali (stimati): {all_datasets['datasets_real'].sum()}")

---
## 3. Metriche per fonte

Calcoliamo le dimensioni che potrebbero comporre un catalog score.

### 3a. Formati aperti
Per CKAN: % di dataset che offrono CSV/JSON (formati tabulari aperti)
Per HTML: % di file CSV/JSON vs ZIP/XLS

In [ ]:
def openness_score(fmt_str, protocol):
    """Stima se un formato è 'aperto' (CSV/JSON) o chiuso."""
    if pd.isna(fmt_str) or fmt_str == "":
        return None
    fmt_lower = fmt_str.lower()
    # Per CKAN: format è "csv,xml" (lista formati disponibili)
    # Per HTML: format è singolo ("CSV", "ZIP", ecc.)
    if protocol == "ckan":
        parts = [f.strip() for f in fmt_lower.split(",")]
        has_csv = "csv" in parts
        has_json = "json" in parts
        return int(has_csv or has_json)
    elif protocol == "html":
        return int(fmt_lower in ("csv", "json"))
    elif protocol == "sparql":
        return 1  # SPARQL endpoint = queryabile
    elif protocol == "sdmx":
        return 1  # SDMX = standard aperto
    return None


df["openness"] = df.apply(lambda r: openness_score(r.get("format"), r["protocol"]), axis=1)

# Media openness per fonte
openness = df.groupby("source_id")["openness"].agg(["mean", "count"]).reset_index()
openness.columns = ["source_id", "openness_ratio", "items_with_format"]
openness["openness_ratio"] = openness["openness_ratio"].fillna(0).round(2)

print("=== Openness ratio (CSV/JSON su tot) ===")
display(openness.sort_values("openness_ratio", ascending=False))

In [ ]:
### 3b. DataStore (per CKAN)
ckan = df[df["protocol"] == "ckan"].copy()
ckan_ds = ckan.groupby("source_id")["datastore_active"].agg(["sum", "count"]).reset_index()
ckan_ds["datastore_ratio"] = (ckan_ds["sum"] / ckan_ds["count"]).round(2)
print("=== DataStore ratio (CKAN) ===")
display(
    ckan_ds[["source_id", "datastore_ratio", "count"]].sort_values(
        "datastore_ratio", ascending=False
    )
)

In [ ]:
### 3c. Varietà organizzativa (proxy di copertura tematica)
orgs = df.groupby("source_id")["organization"].nunique().reset_index(name="n_organizations")
print("=== Varietà organizzativa ===")
display(orgs.sort_values("n_organizations", ascending=False))

In [ ]:
### 3d. Radar health
# Mappa fonte -> radar status
radar_map = {s["id"]: s["status"] for s in radar["sources"]}
df["radar_status"] = df["source_id"].map(radar_map)

radar_status = df.groupby("source_id")["radar_status"].first().reset_index()
radar_status["radar_score"] = radar_status["radar_status"].map(
    {"GREEN": 1.0, "YELLOW": 0.5, "RED": 0.0}
)
print("=== Radar health ===")
display(radar_status.sort_values("radar_score"))

In [ ]:
### 3e. Ricchezza: item_count normalizzato
# Scala log per evitare che ISTAT (4836 item) domini su consip (16 item)
all_datasets["log_datasets"] = np.log10(all_datasets["datasets_real"].clip(lower=1))
print("=== Ricchezza (log10 dataset reali) ===")
display(
    all_datasets[["source_id", "datasets_real", "log_datasets"]].sort_values(
        "log_datasets", ascending=False
    )
)

---
## 4. Composizione Catalog Score (primo prototipo)

Componiamo le metriche in un unico score 0-100.

**Pesi proposti**:
| Dimensione | Peso | Fonte |
|---|---|---|
| Ricchezza (log datasets) | 25 | inventory |
| Openness (CSV/JSON ratio) | 20 | inventory |
| Radar health | 15 | radar |
| DataStore ratio | 15 | inventory (CKAN only) |
| Varietà organizzativa | 15 | inventory |
| Stabilità inventario | 10 | inventory report |

In [ ]:
# Assembliamo tutte le metriche
score_df = all_datasets[["source_id", "datasets_real", "log_datasets"]].copy()

# Openness
score_df = score_df.merge(openness[["source_id", "openness_ratio"]], on="source_id", how="left")
score_df["openness_ratio"] = score_df["openness_ratio"].fillna(0)

# Radar
score_df = score_df.merge(radar_status[["source_id", "radar_score"]], on="source_id", how="left")
score_df["radar_score"] = score_df["radar_score"].fillna(0.5)  # default YELLOW

# DataStore (CKAN) / default per non-CKAN
score_df = score_df.merge(ckan_ds[["source_id", "datastore_ratio"]], on="source_id", how="left")
score_df["datastore_ratio"] = score_df["datastore_ratio"].fillna(0.5)  # default middle

# Organizzazioni
score_df = score_df.merge(orgs[["source_id", "n_organizations"]], on="source_id", how="left")
score_df["n_organizations"] = score_df["n_organizations"].fillna(1)
# Normalizza log organizzazioni
max_org = score_df["n_organizations"].max()
score_df["org_variety"] = (
    np.log10(score_df["n_organizations"].clip(lower=1)) / np.log10(max_org.clip(lower=2))
).round(2)

display(score_df)

In [ ]:
# Normalizzazione componenti a [0, 100]
# Ricchezza: log_datasets -> [0, 100]
max_log = score_df["log_datasets"].max()
score_df["score_ricchezza"] = (score_df["log_datasets"] / max_log * 100).round(1)

# Openness: già 0-1 -> 0-100
score_df["score_openness"] = (score_df["openness_ratio"] * 100).round(1)

# Radar: già 0-1 -> 0-100
score_df["score_radar"] = (score_df["radar_score"] * 100).round(1)

# DataStore: già 0-1 -> 0-100
score_df["score_datastore"] = (score_df["datastore_ratio"] * 100).round(1)

# Varietà organizzativa: già 0-1 -> 0-100
score_df["score_org"] = (score_df["org_variety"] * 100).round(1)

# Pesi
W = {
    "ricchezza": 25,
    "openness": 20,
    "radar": 15,
    "datastore": 15,
    "org": 15,
}

score_df["catalog_score"] = (
    score_df["score_ricchezza"] * W["ricchezza"] / 100
    + score_df["score_openness"] * W["openness"] / 100
    + score_df["score_radar"] * W["radar"] / 100
    + score_df["score_datastore"] * W["datastore"] / 100
    + score_df["score_org"] * W["org"] / 100
).round(1)

# Aggiungiamo la stabilità inventario (10 pt bonus/malus) se abbiamo il report
sig_map = {s["source"]: s for s in signals["signals"]}
stability = []
for sid in score_df["source_id"]:
    sig = sig_map.get(sid, {})
    if sig.get("signal_type") == "inventory change":
        st = 5  # cambiamento = leggermente instabile
    elif sig.get("result") == "skipped":
        st = 3
    else:
        st = 10  # stabile
    stability.append(st)

score_df["score_stability"] = stability
score_df["catalog_score"] = score_df["catalog_score"] + (score_df["score_stability"] * 10 / 100)

# Clamp a 100
score_df["catalog_score"] = score_df["catalog_score"].clip(upper=100)


# Band
def score_band(score):
    if score >= 70:
        return "premium"
    if score >= 40:
        return "solido"
    return "da valutare"


score_df["band"] = score_df["catalog_score"].apply(score_band)

print("=== Catalog Score — Ranking ===")
display(
    score_df.sort_values("catalog_score", ascending=False)[
        ["source_id", "catalog_score", "band", "datasets_real", "openness_ratio", "radar_status"]
    ]
)

In [ ]:
# Distribuzione per banda
print("=== Distribuzione bande ===")
display(score_df["band"].value_counts().to_frame("count"))
print()
print("=== Statistiche score ===")
print(score_df["catalog_score"].describe())

---
## 5. Analisi sensibilità

Lo score cambia molto se togliamo una dimensione? Proviamo.

In [ ]:
# Senza DataStore (non tutte le fonti CKAN lo hanno)
score_no_ds = (
    score_df["score_ricchezza"] * 30 / 100  # redistribuisco peso
    + score_df["score_openness"] * 25 / 100
    + score_df["score_radar"] * 20 / 100
    + score_df["score_org"] * 15 / 100
).round(1)

# Quanto cambia
score_df["score_no_datastore"] = score_no_ds
score_df["delta_no_ds"] = score_df["score_no_datastore"] - score_df["catalog_score"]

print("=== Delta se rimuoviamo DataStore ===")
display(
    score_df[["source_id", "catalog_score", "score_no_datastore", "delta_no_ds"]]
    .sort_values("delta_no_ds")
    .head(10)
)
print("\nDelta medio:", score_df["delta_no_ds"].mean())

---
## 6. Discussione

Osservazioni:

1. **Le dimensioni discriminano bene?** Se tutte le fonti finiscono nella stessa banda,
   lo score non serve. Se spalma le fonti in 3 bande, è utile.

2. **Coerenza interna**: le fonti che sappiamo essere "buone" (ISTAT, INPS, ANAC)
   sono in alto? Quelle deboli (giustizia_statistiche) sono in basso?

3. **Sensibilità alla normalizzazione**: la scala log per ricchezza aiuta
   o appiattisce?

4. **Dipendenza da source_check_results**: quanto manca `dataset_group`
   e `intake_score` per un calcolo più preciso?

5. **Problema DataStore per fonti non-CKAN**: abbiamo usato 0.5 come default.
   Ha senso o è un artefatto?